In [1]:
import math
import qewton

In [2]:
X = qewton.Variable("x", dim=2)
U = qewton.Variable("u", dim=1)

In [3]:
square = qewton.geometries.Rectangle(X, [0.0, 0.0], 1.0, 1.0)

point_sampler = qewton.GridSampler(square, 10000)
points_sampler_boundary = qewton.GridSampler(square.boundary, 5000)

In [4]:
model = qewton.FCN(
    in_neurons=X,
    hidden_neurons=25,
    out_neurons=U,
    n_hidden_layers=4,
    activation=qewton.bb.Tanh,
)

In [5]:
def residual_fun(u: U, x: X):  # type: ignore
    return u.laplacian(x) + 4*math.pi**2*u

pde_graph = qewton.PINNPipeline(point_sampler, [model], residual=residual_fun, residual_name="PINNConstraint")
pde_constraint = pde_graph.constraint

In [6]:
def boundary_residual_fun(u: U, x: X):  # type: ignore
    cos_1 = qewton.bb.Cos()(2*math.pi*x[:, :1])
    cos_2 = qewton.bb.Cos()(2*math.pi*x[:, 1:])
    return u - (cos_1 + cos_2)

boundary_constraint = qewton.constraints.PINNConstraint(
    boundary_residual_fun, name="BoundaryConstraint", weight=1000.0
)
boundary_graph = qewton.PINNPipeline(points_sampler_boundary, [model], boundary_constraint)

In [ ]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=5000,
)

lbfgs_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.LBFGS(),
    lr=0.1,
    max_iterations=1,
    optimizer_args={"max_eval": 100},
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase, lbfgs_phase],
    graphs=[boundary_graph, pde_graph],
    training_objectives=[boundary_constraint, pde_constraint],
    device=qewton.cuda(0),
)

trainer.run()

In [ ]:
predicted_plot = pde_graph.visualize(model.output_ports[0], device=qewton.cuda(0))
result = predicted_plot.evaluate()
coords = result.vertices
predicted = result.color.reshape(-1, 1)

import numpy as np

expected = np.cos(2*math.pi*coords[:, [0]]) + np.cos(2*math.pi*coords[:, [1]])
error = np.abs(predicted - expected)

comparison = np.stack([expected, predicted, error])
print(comparison.shape)
print(expected.shape, predicted.shape, error.shape)
facet_axis = qewton.BatchAxes(3)
geometry = predicted_plot.data_config.geometry_axes.geometry
config = qewton.DataConfiguration(facet_axis, qewton.GeometryAxes(geometry), qewton.FeatureAxes(U))
facet = qewton.visualization.FacetSpec(facet_axis, labels=["Exact", "Predicted", "Absolute Error"])

comparison_plot = qewton.visualization.MeshSurfacePlot(
    comparison, config,
    z=qewton.visualization.AxisSpec(U),
    controls=[facet],
    show_edges=False,
)

fig = qewton.visualization.Figure(comparison_plot)
app = qewton.visualization.DashApplication.create(fig)
app.run(debug=True, jupyter_mode="external")

(3, 441, 1)
(441, 1) (441, 1) (441, 1)
Dash app running on http://127.0.0.1:8050/


Optimization Phase 1:  38%|███▊      | 1877/5000 [00:59<01:16, 40.99it/s, loss=0.67]